# 02 — Historical weather for pilot cities (Day 3)

Download hourly 2021→present weather for **9 pilot cities** (3 per country)
from the Open-Meteo Historical Archive (ERA5, keyless) and validate timestamps.

Validation: exact hourly grid, no gaps/duplicates, tz-aware UTC, identical
grid across cities, no missing values, physical value ranges.

Raw responses are stored verbatim in `data/raw/weather/` with a SHA-256
manifest.

In [ ]:
# Idempotent: existing city files are skipped. On rate-limit (429) the
# module retries with exponential backoff.
from src.ingestion import weather as wx

paths = [wx.download_city(entry) for entry in wx.CITIES]
print(f"\n{len(paths)} city files in {wx.WEATHER_DIR}")

In [ ]:
# Timestamp + value validation for every city.
import pandas as pd

report = wx.validate_all()
rows = []
for stem, res in report["cities"].items():
    rows.append({
        "city": stem, "rows": res["rows"],
        "count_ok": res["row_count_ok"], "tz_utc": res["tz_utc"],
        "monotonic": res["monotonic"], "dups": res["duplicate_times"],
        "gaps": res["gap_count"], "grid_ok": res["grid_matches_expected"],
        "nulls": res["total_nulls"], "out_of_range": res["out_of_range_total"],
    })
display(pd.DataFrame(rows))
print("cross-city grid alignment:", report["grid_alignment_ok"])

In [ ]:
# Quick look: temperature + wind gusts for one city per country.
import matplotlib.pyplot as plt

sample = ["KEN_nairobi", "ZAF_johannesburg", "NGA_lagos"]
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, stem in zip(axes, sample):
    df = wx.load_city(wx.WEATHER_DIR / f"{stem}.json")
    daily = df.set_index("time").resample("D").agg(
        {"temperature_2m": "mean", "wind_gusts_10m": "max"}
    )
    ax.plot(daily.index, daily["temperature_2m"], label="temp °C (daily mean)", lw=0.8)
    ax.set_ylabel(stem)
    ax.legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("UTC date")
plt.tight_layout()
plt.show()

In [ ]:
# Write reports/day03_weather_audit.md
wx.write_audit()

## Day 3 result

| Check | Result |
|---|---|
| Coverage | 9 cities × 50,808 hourly rows (2021-01-01 → 2026-10-18, UTC) |
| Grid | exact hourly grid, 0 gaps, 0 duplicates, all cities aligned |
| Missing values | 0 across all 9 variables |
| Range checks | 0 violations |
| Timestamps | tz-aware UTC by construction; local-time conversion deferred to Day 8 |

Full table: `reports/day03_weather_audit.md`.